Model importing from hugging face

In [20]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### **Install** **libraries**

In [60]:
!pip install -q youtube-transcript-api langchain_text_splitters langchain-community langchain-openai langchain-huggingface \
               faiss-cpu tiktoken python-dotenv

In [31]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

## Step 1a - Indexing (Document Ingestion)

*   List item
*   List item

In [16]:
video_id = "Gfr50f6ZBvo" # only the ID, not full URL
try:
  #If you don't care which language, this returns the "best" one
  transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["en"])

  # Updated version: to handle the raw transcript data you can call fetched_transcript.to_raw_data(), which will return a list of dictionaries:
  updated_transcript=transcript_list.to_raw_data()
  # Flatten it to plain text
  transcript = " ".join(chunk["text"] for chunk in updated_transcript)
  print(transcript)

except TranscriptsDisabled:
  print("No captions available for this video.")

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough 

## Step 1b - Indexing (Text Splitting)

In [17]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [18]:
len(chunks)

168

In [19]:
chunks[100]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [24]:
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True}
)

vector_store = FAISS.from_documents(chunks, embedding)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
vector_store.index_to_docstore_id

{0: 'c61cea9c-1e25-4205-889b-b22dd1f933a8',
 1: '2e51ea42-c9b8-4148-b0ce-768f8aca45f2',
 2: 'ca3b3fca-7723-462a-82d6-c7020f7e9544',
 3: '1ad85d2a-34a4-4d12-a1de-b873256f5a52',
 4: 'f9d318d8-85c3-4bcf-8bcd-49b7e1d095cc',
 5: 'd3380dde-ee00-4fcd-815c-a9b4e322ef5a',
 6: '57133fdd-04a7-4913-876d-e45cf81b9fe2',
 7: '68ae1a70-56b6-41c8-b0ed-ef834a272ef0',
 8: '29074934-cef4-449d-94c6-b6af8f8d299c',
 9: '203d720b-b16d-4dd6-b9f5-5f98d0363ba5',
 10: 'cc9ae3ed-f11b-47fa-90e9-9b657133be20',
 11: 'af48eb0b-f086-4b85-b6db-f66b8ba64f7d',
 12: 'c09524f6-f840-4e09-b476-0930a1fcbe6a',
 13: 'd6922c2e-6cfa-4510-a39c-74d69083734b',
 14: '9d4092b9-8de9-4f50-83b9-5cd7eb6b60ac',
 15: '1b96874b-7ade-41ca-970c-433278079096',
 16: 'ba4515d6-e9da-4119-a494-5fbf0d176021',
 17: '1f2eb497-7b6b-466a-a561-b2ad7c2bd000',
 18: '5ac0322c-7601-4b16-93d8-cc929db38afe',
 19: '3ac8929c-01dc-47a7-a989-6da168402c50',
 20: '5ef6a441-8725-456e-8ee7-0b1a23d1db5a',
 21: '21179eb5-aa8e-4627-a831-9be73171d929',
 22: '6aaa4557-1eac-

In [ ]:
vector_store.index_to_docstore_id

{0: 'c61cea9c-1e25-4205-889b-b22dd1f933a8',
 1: '2e51ea42-c9b8-4148-b0ce-768f8aca45f2',
 2: 'ca3b3fca-7723-462a-82d6-c7020f7e9544',
 3: '1ad85d2a-34a4-4d12-a1de-b873256f5a52',
 4: 'f9d318d8-85c3-4bcf-8bcd-49b7e1d095cc',
 5: 'd3380dde-ee00-4fcd-815c-a9b4e322ef5a',
 6: '57133fdd-04a7-4913-876d-e45cf81b9fe2',
 7: '68ae1a70-56b6-41c8-b0ed-ef834a272ef0',
 8: '29074934-cef4-449d-94c6-b6af8f8d299c',
 9: '203d720b-b16d-4dd6-b9f5-5f98d0363ba5',
 10: 'cc9ae3ed-f11b-47fa-90e9-9b657133be20',
 11: 'af48eb0b-f086-4b85-b6db-f66b8ba64f7d',
 12: 'c09524f6-f840-4e09-b476-0930a1fcbe6a',
 13: 'd6922c2e-6cfa-4510-a39c-74d69083734b',
 14: '9d4092b9-8de9-4f50-83b9-5cd7eb6b60ac',
 15: '1b96874b-7ade-41ca-970c-433278079096',
 16: 'ba4515d6-e9da-4119-a494-5fbf0d176021',
 17: '1f2eb497-7b6b-466a-a561-b2ad7c2bd000',
 18: '5ac0322c-7601-4b16-93d8-cc929db38afe',
 19: '3ac8929c-01dc-47a7-a989-6da168402c50',
 20: '5ef6a441-8725-456e-8ee7-0b1a23d1db5a',
 21: '21179eb5-aa8e-4627-a831-9be73171d929',
 22: '6aaa4557-1eac-

In [ ]:
vector_store.index_to_docstore_id

{0: 'c61cea9c-1e25-4205-889b-b22dd1f933a8',
 1: '2e51ea42-c9b8-4148-b0ce-768f8aca45f2',
 2: 'ca3b3fca-7723-462a-82d6-c7020f7e9544',
 3: '1ad85d2a-34a4-4d12-a1de-b873256f5a52',
 4: 'f9d318d8-85c3-4bcf-8bcd-49b7e1d095cc',
 5: 'd3380dde-ee00-4fcd-815c-a9b4e322ef5a',
 6: '57133fdd-04a7-4913-876d-e45cf81b9fe2',
 7: '68ae1a70-56b6-41c8-b0ed-ef834a272ef0',
 8: '29074934-cef4-449d-94c6-b6af8f8d299c',
 9: '203d720b-b16d-4dd6-b9f5-5f98d0363ba5',
 10: 'cc9ae3ed-f11b-47fa-90e9-9b657133be20',
 11: 'af48eb0b-f086-4b85-b6db-f66b8ba64f7d',
 12: 'c09524f6-f840-4e09-b476-0930a1fcbe6a',
 13: 'd6922c2e-6cfa-4510-a39c-74d69083734b',
 14: '9d4092b9-8de9-4f50-83b9-5cd7eb6b60ac',
 15: '1b96874b-7ade-41ca-970c-433278079096',
 16: 'ba4515d6-e9da-4119-a494-5fbf0d176021',
 17: '1f2eb497-7b6b-466a-a561-b2ad7c2bd000',
 18: '5ac0322c-7601-4b16-93d8-cc929db38afe',
 19: '3ac8929c-01dc-47a7-a989-6da168402c50',
 20: '5ef6a441-8725-456e-8ee7-0b1a23d1db5a',
 21: '21179eb5-aa8e-4627-a831-9be73171d929',
 22: '6aaa4557-1eac-

In [26]:
vector_store.get_by_ids(['7ff5f2aa-4657-4799-aa7d-fc38a71236e5'])

[Document(id='7ff5f2aa-4657-4799-aa7d-fc38a71236e5', metadata={}, page_content="ai we would call it now um people like minsky and and and patrick winston and you know all these characters right and used to debate a few of them and they used to think i was mad thinking about that some new advance could be done with learning systems and um i was actually pleased to hear that because at least you know you're on a unique track at that point right even if every all of your you know professors are telling you you're mad that's true and of course in industry uh you can we couldn't get you know as difficult to get two cents together uh and which is hard to imagine now as well given it's the biggest sort of buzzword in in vcs and and fundraising's easy and all these kind of things today so back in 2010 it was very difficult and what we the reason we started then and shane and i used to discuss um uh uh what were the sort of founding tenets of deep mind and it was very various things one was um 

## Step 2 - Retrieval

In [46]:
retriever = vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [47]:
retriever.invoke("what is deepmind")

[Document(id='f966775f-7de5-46b6-ae8f-58aea0d51576', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

## Step 3 - Augmentation

In [37]:
!hf auth login

User is already logged in. Use `hf auth login --force` to force re-login.


In [39]:
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=256,
    temperature=0.2,
    device=0
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [61]:
llm = HuggingFacePipeline(pipeline=pipe)

In [48]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [49]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [50]:
retrieved_docs

[Document(id='070b3dea-2d1d-433f-8b02-850c2deee415', metadata={}, page_content="so we with this problem and we published it in a nature paper last year uh we held the fusion that we held the plasma in specific shapes so actually it's almost like carving the plasma into different shapes and control and hold it there for the record amount of time so um so that's one of the problems of of fusion sort of um solved so i have a controller that's able to no matter the shape uh contain it continue yeah contain it and hold it in structure and there's different shapes that are better for for the energy productions called droplets and and and so on so um so that was huge and now we're looking we're talking to lots of fusion startups to see what's the next problem we can tackle uh in the fusion area so another fascinating place in a paper title pushing the frontiers of density functionals by solving the fractional electron problem so you're taking on modeling and simulating the quantum mechanical 

In [51]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"so we with this problem and we published it in a nature paper last year uh we held the fusion that we held the plasma in specific shapes so actually it's almost like carving the plasma into different shapes and control and hold it there for the record amount of time so um so that's one of the problems of of fusion sort of um solved so i have a controller that's able to no matter the shape uh contain it continue yeah contain it and hold it in structure and there's different shapes that are better for for the energy productions called droplets and and and so on so um so that was huge and now we're looking we're talking to lots of fusion startups to see what's the next problem we can tackle uh in the fusion area so another fascinating place in a paper title pushing the frontiers of density functionals by solving the fractional electron problem so you're taking on modeling and simulating the quantum mechanical behavior of electrons yes um can you explain this work and can ai model and\n\n

In [52]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [53]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      so we with this problem and we published it in a nature paper last year uh we held the fusion that we held the plasma in specific shapes so actually it's almost like carving the plasma into different shapes and control and hold it there for the record amount of time so um so that's one of the problems of of fusion sort of um solved so i have a controller that's able to no matter the shape uh contain it continue yeah contain it and hold it in structure and there's different shapes that are better for for the energy productions called droplets and and and so on so um so that was huge and now we're looking we're talking to lots of fusion startups to see what's the next problem we can tackle uh in the fusion area so another fascinating place in a paper title pushing the frontiers of density functionals

## Step 4 - Generation

In [64]:
answer = llm.invoke(final_prompt)

In [65]:
print(answer)


      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      so we with this problem and we published it in a nature paper last year uh we held the fusion that we held the plasma in specific shapes so actually it's almost like carving the plasma into different shapes and control and hold it there for the record amount of time so um so that's one of the problems of of fusion sort of um solved so i have a controller that's able to no matter the shape uh contain it continue yeah contain it and hold it in structure and there's different shapes that are better for for the energy productions called droplets and and and so on so um so that was huge and now we're looking we're talking to lots of fusion startups to see what's the next problem we can tackle uh in the fusion area so another fascinating place in a paper title pushing the frontiers of density functionals by solving the fractional el

## Building a Chain

In [66]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [67]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [68]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [69]:
parallel_chain.invoke('who is Demis')

{'context': "the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get

In [70]:
parser = StrOutputParser()

In [71]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')